In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print("Imports done ✅")

Imports done ✅


In [2]:
ROOT       = Path.cwd().parent
TRANS_PATH = ROOT / 'data' / 'raw' / 'train_transaction.csv'
IDENT_PATH = ROOT / 'data' / 'raw' / 'train_identity.csv'

print(f"Transaction file exists: {TRANS_PATH.exists()}")
print(f"Identity file exists   : {IDENT_PATH.exists()}")

Transaction file exists: True
Identity file exists   : True


In [ ]:
# ── Load both tables ──────────────────────────────────────────────────────────
# train_transaction: 590k rows, 394 features + target (isFraud)
# train_identity: identity info for ~144k of those transactions
# They join on TransactionID

print("Loading transaction table...")
trans = pd.read_csv(TRANS_PATH)
print(f"  Shape: {trans.shape}")

print("Loading identity table...")
ident = pd.read_csv(IDENT_PATH)
print(f"  Shape: {ident.shape}")

Loading transaction table...


In [ ]:
# ── Left join — keep all transactions, add identity where available ────────────
# Not every transaction has identity info — that's expected in real fraud data.
# Transactions without identity info get NaN for identity columns.

df = trans.merge(ident, on='TransactionID', how='left')

print(f"After merge shape: {df.shape}")
print(f"Transactions with identity info: {ident.shape[0]:,} / {trans.shape[0]:,} "
      f"({ident.shape[0]/trans.shape[0]*100:.1f}%)")

In [ ]:
print("── Columns ────────────────────────────────────────────")
print(f"Total columns: {len(df.columns)}")
print("\nFirst 20 columns:")
print(df.columns[:20].tolist())

print("\n── Target distribution ─────────────────────────────────")
fraud_counts = df['isFraud'].value_counts()
fraud_pct    = df['isFraud'].value_counts(normalize=True) * 100

print(f"  Legitimate (0): {fraud_counts[0]:>8,}  ({fraud_pct[0]:.2f}%)")
print(f"  Fraud      (1): {fraud_counts[1]:>8,}  ({fraud_pct[1]:.2f}%)")
print(f"\n  Imbalance ratio: {fraud_counts[0]//fraud_counts[1]}:1")

In [ ]:
# ── Q1: How severe is the class imbalance? ────────────────────────────────────
# This is the core challenge of fraud detection.
# A model that predicts "not fraud" for everything gets 99.8% accuracy — useless.
# This is why we can NEVER use accuracy as our metric here.

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
colors = ['steelblue', 'crimson']
axes[0].bar(['Legitimate', 'Fraud'], fraud_counts.values, color=colors, edgecolor='none')
axes[0].set_title('Transaction Counts', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Percentage pie
axes[1].pie(fraud_counts.values, labels=['Legitimate', 'Fraud'],
            colors=colors, autopct='%1.2f%%', startangle=90)
axes[1].set_title('Class Distribution', fontweight='bold')

plt.suptitle('Q1: Class Imbalance — The Core Challenge', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../notebooks/class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

print("Decision: NEVER use accuracy. Use Precision, Recall, F1, AUC-ROC.")
print("Decision: Apply class_weight='balanced' or SMOTE to handle imbalance.")

In [ ]:
# ── Q2: How much data is missing? ────────────────────────────────────────────
# 434 features — many will be sparse. We need a clear cutoff for dropping columns.

missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct':   missing_pct
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

print(f"Columns with missing values: {len(missing_df)} / {len(df.columns)}")
print(f"\nMissingness distribution:")
print(f"  > 90% missing: {(missing_pct > 90).sum()} columns")
print(f"  > 50% missing: {(missing_pct > 50).sum()} columns")
print(f"  > 20% missing: {(missing_pct > 20).sum()} columns")
print(f"  < 20% missing: {(missing_pct < 20).sum()} columns")

# Plot top 30 most missing columns
top_missing = missing_df.head(30)

plt.figure(figsize=(12, 7))
plt.barh(top_missing.index, top_missing['missing_pct'],
         color='steelblue', edgecolor='none')
plt.axvline(50, color='red', linestyle='--', linewidth=1.5, label='50% threshold')
plt.xlabel('Missing %')
plt.title('Q2: Top 30 Columns by Missing Data', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../notebooks/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Decision: drop columns above 50% missing ─────────────────────────────────
# Columns above 50% missing are too sparse for reliable imputation.
# Keeping them adds noise and slows training with no benefit.

DROP_THRESHOLD = 50.0

cols_to_drop = missing_pct[missing_pct > DROP_THRESHOLD].index.tolist()
print(f"Columns to drop (>{DROP_THRESHOLD}% missing): {len(cols_to_drop)}")
print(f"Columns to keep: {len(df.columns) - len(cols_to_drop)}")

In [ ]:
# ── Q3: How does transaction amount differ between fraud and legitimate? ───────
# Amount is often the strongest single predictor in fraud detection.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

legit = df[df['isFraud'] == 0]['TransactionAmt']
fraud = df[df['isFraud'] == 1]['TransactionAmt']

# Log-scale distribution
axes[0].hist(np.log1p(legit), bins=80, alpha=0.6,
             color='steelblue', label='Legitimate', edgecolor='none')
axes[0].hist(np.log1p(fraud), bins=80, alpha=0.6,
             color='crimson', label='Fraud', edgecolor='none')
axes[0].set_title('Transaction Amount Distribution (Log Scale)', fontweight='bold')
axes[0].set_xlabel('log(Transaction Amount)')
axes[0].legend()

# Box plot
df_plot = df[['TransactionAmt', 'isFraud']].copy()
df_plot['isFraud'] = df_plot['isFraud'].map({0: 'Legitimate', 1: 'Fraud'})
df_plot['log_amt'] = np.log1p(df_plot['TransactionAmt'])
axes[1].boxplot(
    [np.log1p(legit), np.log1p(fraud)],
    labels=['Legitimate', 'Fraud'],
    patch_artist=True,
    boxprops=dict(facecolor='steelblue', alpha=0.6)
)
axes[1].set_title('Amount Box Plot (Log Scale)', fontweight='bold')
axes[1].set_ylabel('log(Amount)')

plt.suptitle('Q3: Transaction Amount — Fraud vs Legitimate', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../notebooks/amount_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Median legitimate amount: ${legit.median():.2f}")
print(f"Median fraud amount     : ${fraud.median():.2f}")

In [ ]:
# ── Q4: Which product codes have the highest fraud rates? ────────────────────
# ProductCD is one of the most important categorical features.

fraud_by_product = df.groupby('ProductCD')['isFraud'].agg(['mean', 'sum', 'count'])
fraud_by_product.columns = ['fraud_rate', 'fraud_count', 'total']
fraud_by_product['fraud_rate_pct'] = fraud_by_product['fraud_rate'] * 100
fraud_by_product = fraud_by_product.sort_values('fraud_rate_pct', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(fraud_by_product.index, fraud_by_product['fraud_rate_pct'],
             color='steelblue', edgecolor='none')
axes[0].set_title('Fraud Rate by Product Code', fontweight='bold')
axes[0].set_ylabel('Fraud Rate (%)')

axes[1].bar(fraud_by_product.index, fraud_by_product['total'],
             color='coral', edgecolor='none', label='Total')
axes[1].bar(fraud_by_product.index, fraud_by_product['fraud_count'],
             color='crimson', edgecolor='none', label='Fraud')
axes[1].set_title('Transaction Volume by Product Code', fontweight='bold')
axes[1].legend()

plt.suptitle('Q4: Fraud Rate by Product Code', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../notebooks/fraud_by_product.png', dpi=150, bbox_inches='tight')
plt.show()

print(fraud_by_product[['fraud_rate_pct', 'fraud_count', 'total']].to_string())

In [ ]:
# ── Q5: Does fraud frequency change over time? ────────────────────────────────
# TransactionDT is seconds from a reference date — not a real timestamp.
# We can still look at trends across the dataset time range.

df['day'] = df['TransactionDT'] // 86400   # convert seconds to days

daily = df.groupby('day')['isFraud'].agg(['mean', 'count'])
daily.columns = ['fraud_rate', 'total_transactions']

fig, ax1 = plt.subplots(figsize=(13, 4))

color_fr  = 'crimson'
color_vol = 'steelblue'

ax1.plot(daily.index, daily['fraud_rate'] * 100,
          color=color_fr, linewidth=1.5, alpha=0.8)
ax1.set_xlabel('Day')
ax1.set_ylabel('Daily Fraud Rate (%)', color=color_fr)
ax1.tick_params(axis='y', labelcolor=color_fr)

ax2 = ax1.twinx()
ax2.bar(daily.index, daily['total_transactions'],
         color=color_vol, alpha=0.2)
ax2.set_ylabel('Transaction Volume', color=color_vol)
ax2.tick_params(axis='y', labelcolor=color_vol)

plt.title('Q5: Fraud Rate & Volume Over Time', fontweight='bold')
plt.tight_layout()
plt.savefig('../notebooks/fraud_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## EDA Summary & Modeling Decisions

| Finding | Decision |
|---|---|
| 3.5% fraud rate — severe imbalance | Use class_weight='balanced' + evaluate with F1/AUC, not accuracy |
| ~200 columns >50% missing | Drop columns above 50% missing threshold |
| Fraud amounts differ from legitimate | Include TransactionAmt + log transform |
| ProductCD has strong fraud signal | Include as categorical feature |
| TransactionDT shows time trends | Extract day feature |
| 434 raw features | Use feature importance to select top features after first model |

## Why accuracy is useless here
A model predicting "not fraud" on everything scores 96.5% accuracy
but catches 0 fraud cases. We optimize for:
- **Recall** — catching as many frauds as possible (false negatives = missed fraud = money lost)
- **Precision** — not flagging too many legit transactions (false positives = customer frustration)
- **F1** — balance of both
- **AUC-ROC** — overall discrimination ability